# Detailed Algorithm Comparison

Use this notebook when you want a current comparison recipe instead of the old AutoNSGA-II-style examples. It keeps the setup aligned with the public `optimize(...)` API and the current algorithm catalog.


In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vamos import optimize
from vamos.algorithms import available_algorithms
from vamos.foundation.quality_indicators import compute_hypervolume

print("Available algorithms:", available_algorithms())


## Bi-objective algorithms on ZDT1

These algorithms are easy to compare on the same 2-objective front using the same evaluation budget.


In [ ]:
BI_CASES = [
    ("nsgaii", {"problem": "zdt1", "max_evaluations": 10000, "pop_size": 100}),
    ("spea2", {"problem": "zdt1", "max_evaluations": 10000, "pop_size": 100}),
    ("smsemoa", {"problem": "zdt1", "max_evaluations": 10000, "pop_size": 100}),
    ("ibea", {"problem": "zdt1", "max_evaluations": 10000, "pop_size": 100}),
    ("smpso", {"problem": "zdt1", "max_evaluations": 10000, "pop_size": 100}),
]


def run_case(algorithm: str, *, problem: str, max_evaluations: int, pop_size: int, n_obj: int | None = None):
    start = time.perf_counter()
    result = optimize(
        problem,
        algorithm=algorithm,
        max_evaluations=max_evaluations,
        pop_size=pop_size,
        n_obj=n_obj,
        seed=42,
        engine="numpy",
    )
    elapsed = time.perf_counter() - start
    return result, elapsed


bi_rows = []
bi_results = {}
ref_point_2d = np.array([1.1, 1.1])

for algorithm, kwargs in BI_CASES:
    result, elapsed = run_case(algorithm, **kwargs)
    bi_results[algorithm] = result
    hv = compute_hypervolume(result.F, ref_point_2d)
    bi_rows.append(
        {
            "algorithm": algorithm,
            "solutions": len(result),
            "hv": hv,
            "runtime_s": elapsed,
        }
    )

bi_df = pd.DataFrame(bi_rows).sort_values("hv", ascending=False)
bi_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, result in bi_results.items():
    axes[0].scatter(result.F[:, 0], result.F[:, 1], s=20, alpha=0.6, label=name.upper())

f1_true = np.linspace(0, 1, 200)
f2_true = 1 - np.sqrt(f1_true)
axes[0].plot(f1_true, f2_true, "k--", linewidth=2, alpha=0.5, label="ZDT1 PF")
axes[0].set_title("ZDT1 fronts")
axes[0].set_xlabel("f1")
axes[0].set_ylabel("f2")
axes[0].legend(loc="best")
axes[0].grid(True, alpha=0.3)

axes[1].bar(bi_df["algorithm"].str.upper(), bi_df["runtime_s"])
axes[1].set_title("Runtime")
axes[1].set_ylabel("seconds")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


## Many-objective algorithms on DTLZ2

These algorithms are designed for 3+ objectives and are better judged on a many-objective benchmark.


In [ ]:
MANY_CASES = [
    ("moead", {"problem": "dtlz2", "max_evaluations": 12000, "pop_size": 91, "n_obj": 3}),
    ("nsgaiii", {"problem": "dtlz2", "max_evaluations": 12000, "pop_size": 92, "n_obj": 3}),
    ("agemoea", {"problem": "dtlz2", "max_evaluations": 12000, "pop_size": 91, "n_obj": 3}),
    ("rvea", {"problem": "dtlz2", "max_evaluations": 12000, "pop_size": 91, "n_obj": 3}),
]

many_rows = []
ref_point_3d = np.array([1.2, 1.2, 1.2])

for algorithm, kwargs in MANY_CASES:
    result, elapsed = run_case(algorithm, **kwargs)
    hv = compute_hypervolume(result.F, ref_point_3d)
    many_rows.append(
        {
            "algorithm": algorithm,
            "solutions": len(result),
            "hv": hv,
            "runtime_s": elapsed,
        }
    )

many_df = pd.DataFrame(many_rows).sort_values("hv", ascending=False)
many_df


## Reading the comparison

- Use the ZDT1 section to compare 2-objective algorithms and swarm/indicator methods.
- Use the DTLZ2 section to compare many-objective algorithms.
- For serious studies, increase budgets, run multiple seeds, and record the resolved configuration and environment. `30_paper_benchmarking.ipynb` shows how to inspect retained benchmark data.
